In [20]:
import importlib
import warnings
from pathlib import Path
import sys

import pandas as pd

warnings.filterwarnings("ignore")

sys.path.append("../")

import src.train as train_module
import src.predict as predict_module

importlib.reload(train_module)
importlib.reload(predict_module)

run_cv = train_module.run_cv
train_and_save = train_module.train_and_save
predict_from_dataframe = predict_module.predict_from_dataframe

In [10]:
params = {
    "random_state": 42,
    "bagging_seed": 42,
    "feature_fraction_seed": 42,
    "data_random_seed": 42,
    "deterministic": True,
    "force_col_wise": True,
    "n_estimators": 5000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}

In [11]:
BASE_DIR = Path("../data/raw")
train_df = pd.read_csv(BASE_DIR / "train.csv")
test_df = pd.read_csv(BASE_DIR / "test.csv")

print(train_df.shape, test_df.shape)
train_df.head()

(594194, 21) (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [12]:
categorical_cols = list(train_df.select_dtypes(include=["object", "category"]).columns)
if "SeniorCitizen" in train_df.columns:
    categorical_cols.append("SeniorCitizen")

categorical_cols = sorted(set(categorical_cols) - {"Churn"})
print(f"categorical features: {len(categorical_cols)}")
print(categorical_cols)

categorical features: 16
['Contract', 'Dependents', 'DeviceProtection', 'InternetService', 'MultipleLines', 'OnlineBackup', 'OnlineSecurity', 'PaperlessBilling', 'Partner', 'PaymentMethod', 'PhoneService', 'SeniorCitizen', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'gender']


In [21]:
oof, metrics, oof_df = run_cv(
    train_df,
    target_col="Churn",
    model_type="lgb",
    params=params,
    n_splits=4,
    seed=42,
    categorical_cols=categorical_cols,
    id_cols=["id"],
    return_oof_frame=True,
    threshold=0.5,
)

print(f"CV AUC mean: {sum(metrics)/len(metrics):.5f}")
oof_df.head()


Fold 1
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 100363, number of negative: 345282
[LightGBM] [Info] Total Bins 892
[LightGBM] [Info] Number of data points in the train set: 445645, number of used features: 31
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225208 -> initscore=-1.235568
[LightGBM] [Info] Start training from score -1.235568
AUC: 0.9167

Fold 2
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 100362, number of negative: 345283
[LightGBM] [Info] Total Bins 892
[LightGBM] [Info] Number of data points in the train set: 445645, number of used features: 31
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225206 -> initscore=-1.235581
[LightGBM] [Info] Start training from 

,id,row_index,fold,y_true,oof_pred,oof_pred_label
0,0,0,4,0,0.011587,0
1,1,1,3,0,0.001260,0
2,2,2,3,0,0.214853,0
3,3,3,1,1,0.810887,1
4,4,4,3,1,0.796281,1


In [14]:
model_dir = Path("../models/lgbm_baseline")
_, metadata = train_and_save(
    train_df,
    target_col="Churn",
    model_type="lgb",
    params=params,
    output_dir=model_dir,
    categorical_cols=categorical_cols,
    id_cols=["id"],
)

print(f"saved model: {model_dir / 'model.pkl'}")
print(f"saved metadata: {model_dir / 'metadata.json'}")
metadata["model_type"], len(metadata["preprocessing"]["feature_columns"])

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 133817, number of negative: 460377
[LightGBM] [Info] Total Bins 892
[LightGBM] [Info] Number of data points in the train set: 594194, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225208 -> initscore=-1.235573
[LightGBM] [Info] Start training from score -1.235573
saved model: ..\models\lgbm_baseline\model.pkl
saved metadata: ..\models\lgbm_baseline\metadata.json


('lgb', 31)

In [15]:
pred_df = predict_from_dataframe(test_df, model_dir=model_dir, threshold=0.5)

output_path = Path("../data/processed/test_predictions.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(output_path, index=False)

print(f"saved predictions: {output_path}")
pred_df.head()

saved predictions: ..\data\processed\test_predictions.csv


,id,churn_proba,churn_pred
0,594194,0.111140,0
1,594195,0.000091,0
2,594196,0.084847,0
3,594197,0.003965,0
4,594198,0.394433,0


In [26]:
oof_df.shape
oof_df[["oof_pred", "y_true"]].describe().T

,count,mean,std,min,25%,50%,75%,max
oof_pred,594194.0,0.225164,0.281208,0.000077,0.006661,0.069063,0.409941,0.987553
y_true,594194.0,0.225208,0.417719,0.000000,0.000000,0.000000,0.000000,1.000000


In [29]:
oof_df.head()

#  間違ってるdata を確認する
error_df = oof_df[oof_df["oof_pred_label"] != oof_df["y_true"]]

,id,row_index,fold,y_true,oof_pred,oof_pred_label
15,15,15,2,0,0.612551,1
67,67,67,4,0,0.621599,1
83,83,83,2,0,0.566457,1
85,85,85,3,0,0.931546,1
88,88,88,3,0,0.560664,1
...,...,...,...,...,...,...
666,666,666,2,1,0.301648,0
669,669,669,1,1,0.182267,0
671,671,671,4,0,0.846745,1
673,673,673,1,1,0.315807,0
